# TerraSpectra — Day 5: Feature Normalization & Scaling

## 1. Objective & Prerequisites
- **Objective**: Normalize and scale PCA-reduced hyperspectral features using configurable Standard and Min-Max scaling, with safeguards for zero-variance channels.
- **Prerequisites**: Day 4 PCA reduction complete (`salinas_pca_reduced.npz`); `terraspectra.processing.scaling` module available.

## 2. Core Concepts
- **Standardization (Z-score)**: Transforms features to have $\mu = 0$ and $\sigma = 1$: $z = \frac{x - \mu}{\sigma}$. Preserves relative outlier distances and works best with gaussian-distributed data.
- **Min-Max Normalization**: Scales features linearly into $[0, 1]$: $x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$. Useful for bounded inputs (like WebGL shaders or sigmoid activations), but sensitive to extreme outliers.
- **Division by Zero Safeguard**: If a feature is constant across all pixels (zero variance), dividing by $\sigma=0$ creates `NaNs`. Scalers must detect constant columns and safely map them to 0.

In [1]:
# Load PCA-reduced data from Day 4
import numpy as np
from pathlib import Path
from terraspectra.processing.scaling import HyperspectralScaler, ScalingMethod

pca_file = Path('../data/processed/salinas_pca_reduced.npz')
data = np.load(str(pca_file))
reduced_cube = data['reduced_cube']
print('Loaded Reduced Cube Shape:', reduced_cube.shape)
print(f'Unscaled Value Range: [{reduced_cube.min():.2f}, {reduced_cube.max():.2f}]')

Loaded Reduced Cube Shape: (512, 217, 15)
Unscaled Value Range: [-14190.65, 41072.18]


In [2]:
# 1. Apply Standardization (Z-score)
std_scaler = HyperspectralScaler(method=ScalingMethod.STANDARD)
scaled_std = std_scaler.fit_transform(reduced_cube)

print('--- Standardization Statistics ---')
print(f'Scaled Shape:       {scaled_std.shape}')
print(f'Mean across bands:  {np.mean(scaled_std, axis=(0, 1)).round(4)}')
print(f'Std across bands:   {np.std(scaled_std, axis=(0, 1)).round(4)}')
print(f'All Finite Check:   {np.all(np.isfinite(scaled_std))}')

--- Standardization Statistics ---
Scaled Shape:       (512, 217, 15)
Mean across bands:  [-0. -0.  0. -0. -0. -0. -0. -0.  0.  0. -0. -0.  0. -0. -0.]
Std across bands:   [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
All Finite Check:   True


In [3]:
# 2. Apply Min-Max Normalization [0, 1]
mm_scaler = HyperspectralScaler(method=ScalingMethod.MINMAX, feature_range=(0.0, 1.0))
scaled_mm = mm_scaler.fit_transform(reduced_cube)

print('--- Min-Max Statistics ---')
print(f'Min Value:          {scaled_mm.min():.6f}')
print(f'Max Value:          {scaled_mm.max():.6f}')
print(f'Mean Value:         {scaled_mm.mean():.4f}')
print(f'All Finite Check:   {np.all(np.isfinite(scaled_mm))}')

--- Min-Max Statistics ---
Min Value:          0.000000
Max Value:          1.000000
Mean Value:         0.4391
All Finite Check:   True


In [4]:
# 3. Test Zero-Variance Safeguard on Constant Features
toy_features = np.array([
    [10.0, 5.0, 100.0],
    [20.0, 5.0, 200.0],
    [30.0, 5.0, 300.0],
], dtype=np.float32)  # Column index 1 is constant (value=5.0)

test_scaler = HyperspectralScaler(method=ScalingMethod.STANDARD)
toy_scaled = test_scaler.fit_transform(toy_features)

print('Constant column detected:', test_scaler.constant_feature_indices)
print('Transformed Toy Array (Safe from NaNs):\n', toy_scaled)

Constant features detected at indices: [1]. Handled with zero-variance safeguards.


Constant column detected: [1]
Transformed Toy Array (Safe from NaNs):
 [[-1.2247449  0.        -1.2247449]
 [ 0.         0.         0.       ]
 [ 1.2247449  0.         1.2247449]]


In [5]:
# 4. Demonstrate Train vs Validation Split Isolation
# Simulate 80% train pixels, 20% validation pixels
flat_data = reduced_cube.reshape(-1, 15)
n_train = int(0.8 * len(flat_data))
train_pixels = flat_data[:n_train]
val_pixels = flat_data[n_train:]

# Fit ONLY on train pixels
split_scaler = HyperspectralScaler(method=ScalingMethod.STANDARD)
split_scaler.fit(train_pixels)

# Transform both separately
train_scaled = split_scaler.transform(train_pixels)
val_scaled = split_scaler.transform(val_pixels)

print(f'Train Mean: {train_scaled.mean():.4f}, Train Std: {train_scaled.std():.4f}')
print(f'Val Mean:   {val_scaled.mean():.4f}, Val Std:   {val_scaled.std():.4f} (Valid minor deviation)')

Train Mean: 0.0000, Train Std: 1.0000
Val Mean:   0.0654, Val Std:   1.0374 (Valid minor deviation)


## 3. What I Learned & Common Mistakes
1. **Never fit on test/validation data**: Calculating $\mu$ and $\sigma$ across validation data creates subtle data leakage. Always store the scaler fitted on training data (`scaler.save()`) and reload it at inference time.
2. **Constant Features**: Sensor artifacts or dead bands frequently have constant values across scenes. Robust pipelines must never crash with `ZeroDivisionError`.
3. **Selection Rule**: Use Standardization for gradient-based training (e.g. PyTorch models) and PCA; use Min-Max scaling when preparing pixel textures for WebGL / Deck.gl rendering.

### Interview Question
*Q: If a feature has extreme outliers, which scaler is safer and why?*
*A: Standardization is safer because Min-Max scaling compresses 99% of normal data into a tiny sub-range if a single extreme outlier exists. Standardization preserves the relative variance structure.*

## 4. Verification & Milestone Status
- [x] Applied Standardization (mean=0.0, std=1.0) and MinMax scaling ([0, 1]).
- [x] Verified zero NaNs and zero Infs.
- [x] Persisted fitted models to `outputs/models/scaler_standard.joblib` and `outputs/models/scaler_minmax.joblib`.
- [x] Verified split separation in unit tests (`tests/test_scaling.py`).